Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


Load D5 Dataset

In [2]:
df = pd.read_csv("../datasets/raw/IndianWeatherRepository.csv")
features_to_remove = [
    "country",
    "temperature_fahrenheit",
    "wind_mph",
    "pressure_in",
    "precip_in",
    "visibility_miles",
    "gust_mph"
]
df = df.drop(columns=features_to_remove)
df["last_updated"] = pd.to_datetime(
    df["last_updated"],
    format="%d-%m-%y %H:%M"
)
df["year"] = df["last_updated"].dt.year
df["month"] = df["last_updated"].dt.month
df["day"] = df["last_updated"].dt.day
df["hour"] = df["last_updated"].dt.hour
df["weekday"] = df["last_updated"].dt.dayofweek

df["sunrise"] = pd.to_datetime(
    df["sunrise"],
    format="%I:%M %p"
)
df["sunset"] = pd.to_datetime(
    df["sunset"],
    format="%I:%M %p"
)
df["day_length_minutes"] = (
    (df["sunset"] - df["sunrise"])
    .dt.total_seconds() / 60
)
df["temperature_gap"] = (
    df["feels_like_celsius"]
    - df["temperature_celsius"]
)
df["pm_difference"] = (
    df["air_quality_PM10"]
    - df["air_quality_PM2.5"]
)
df["pollution_intensity"] = (
    df["air_quality_PM2.5"]
    + df["air_quality_PM10"]
    + df["air_quality_Carbon_Monoxide"]
)
df["wind_humidity_interaction"] = (
    df["wind_kph"]
    * df["humidity"]
)
df["humidity_cloud_interaction"] = (
    df["humidity"]
    * df["cloud"]
)
df["heatwave_index"] = (
    df["temperature_celsius"]
    * df["uv_index"]
)


Recreate Categories

In [3]:
def get_day_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"
df["day_period"] = df["hour"].apply(get_day_period)
def get_season(month):
    if month in [3,4,5]:
        return "Summer"
    elif month in [6,7,8,9]:
        return "Monsoon"
    elif month in [10,11]:
        return "Post_Monsoon"
    else:
        return "Winter"
df["season"] = df["month"].apply(get_season)


Recreate Categories

In [4]:
df = df.drop(columns=["last_updated", "sunrise", "sunset"])

Drop Moonrise / Moonset

In [ ]:
df = df.drop(columns=["moonrise", "moonset"])

Identify Categorical Features

In [6]:
categorical_columns = df.select_dtypes(include=["object"]).columns
print(categorical_columns)

C:\Users\ANAMIKA\AppData\Local\Temp\ipykernel_9640\2156690537.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


Index(['location_name', 'region', 'timezone', 'condition_text', 'wind_direction', 'moon_phase', 'day_period', 'season'], dtype='str')


Label Encoding

In [7]:
encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

Verify Dataset

In [ ]:
print(df.shape)
print(df.select_dtypes(include=["object"]).columns)

(119398, 44)
Index([], dtype='str')


Scale Features

In [9]:
scaler = RobustScaler()
scaled_df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

Final Verification

In [10]:
print("Original Shape:", df.shape)
print("Scaled Shape:", scaled_df.shape)
print(scaled_df.head())

Original Shape: (119398, 44)
Scaled Shape: (119398, 44)
   location_name  region  latitude  longitude  timezone  last_updated_epoch  temperature_celsius  condition_text  wind_kph  wind_degree  wind_direction  pressure_mb  precip_mm  humidity  cloud  \
0      -0.902174     0.0  0.095385  -0.123883       0.0           -0.997256             0.759036          1.8750  2.245902     0.720183        1.285714    -0.833333        0.0  0.097561  0.450   
1       0.514493     0.0 -0.095385  -0.113665       0.0           -0.997256             0.759036          2.8125  1.426230     0.747706        1.285714    -0.833333        0.0  0.170732  0.275   
2      -0.608696     0.0 -0.289231   0.030651       0.0           -0.997256             0.614458          1.8750  1.901639     0.885321        0.285714    -0.666667        0.0  0.170732  1.075   
3      -0.760870     0.0 -0.321538  -0.097063       0.0           -0.997256             0.530120          0.0625  1.655738     0.793578        1.285714    -0.66

Final Report

In [11]:
print("""
FINAL PREPROCESSING REPORT
Completed:
1. Feature Removal
2. Datetime Processing
3. Feature Engineering
4. Datetime Cleanup
5. Moonrise/Moonset Removal
6. Label Encoding
7. Robust Scaling
Dataset ready for:
1. KMeans Clustering
2. Isolation Forest
3. Rainfall Prediction
4. Heatwave Prediction
5. Climate Risk Engine
""")


FINAL PREPROCESSING REPORT
Completed:
1. Feature Removal
2. Datetime Processing
3. Feature Engineering
4. Datetime Cleanup
5. Moonrise/Moonset Removal
6. Label Encoding
7. Robust Scaling
Dataset ready for:
1. KMeans Clustering
2. Isolation Forest
3. Rainfall Prediction
4. Heatwave Prediction
5. Climate Risk Engine

